In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/bench/checkpoints/pre_cell_10.pickle

In [4]:
%%RecordEvent
%%time
### cell 10 ###

### cell 10 (vectorized cuDF version)

# Combines percent distributions across years, fully on GPU via cudf.pandas

def combine_subset_of_data_from_multiple_years_22(question, x_axis_title, include_2017=None):
    # unified gender mapping
    gender_map_all = {
        'Male': 'Man',
        'Female': 'Woman',
        'Nonbinary': 'Prefer to self-describe',
        'Prefer not to say': 'Prefer to self-describe',
        'A different identity': 'Prefer to self-describe',
        'Non-binary, genderqueer, or gender non-conforming': 'Prefer to self-describe'
    }
    # map each year to its DataFrame
    df_map = {
        2022: responses_df_2022_cell10,
        2021: responses_df_2021,
        2020: responses_df_2020,
        2019: responses_df_2019_cell10,
        2018: responses_df_2018_cell10,
        2017: responses_df_2017
    }
    years = [2022, 2021, 2020, 2019, 2018]
    if include_2017:
        years.append(2017)

    dfs = []
    for yr in years:
        df = df_map[yr]
        # select the right column in 2017
        col = 'GenderSelect' if (question == 'What is your gender? - Selected Choice' and yr == 2017) else question
        # wrap into cuDF DataFrame via pandas API
        tmp = pd.DataFrame(df[[col]].rename(columns={col: x_axis_title}))
        tmp['year'] = str(yr)
        dfs.append(tmp)

    # concatenate using pandas API (cudf.pandas will dispatch to cudf.concat)
    combined = pd.concat(dfs, ignore_index=True)

    # apply gender normalization
    if question == 'What is your gender? - Selected Choice':
        combined[x_axis_title] = combined[x_axis_title].replace(gender_map_all)

    # compute counts and percentages in one pass
    counts = (
        combined
        .groupby(['year', x_axis_title], sort=False)
        .size()
        .reset_index(name='count')
    )
    totals = counts.groupby('year')['count'].transform('sum')
    counts['percentage'] = (counts['count'] / totals * 100).round(1)

    # select and sort
    result = counts[['percentage', 'year', x_axis_title]]
    return result.sort_values(by=['year', 'percentage'])

# call
question_of_interest_cell22 = 'What is your gender? - Selected Choice'
title_for_x_axis_cell22 = ''
age_df_combined_cell22 = combine_subset_of_data_from_multiple_years_22(
    question_of_interest_cell22,
    title_for_x_axis_cell22,
    include_2017=True
)
age_df_combined_cell22 = age_df_combined_cell22.sort_values(by=['year', 'percentage'])
age_df_combined_cell22.info()

<class 'cudf.core.dataframe.DataFrame'>
Index: 18 entries, 15 to 0
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   percentage  18 non-null     float64
 1   year        18 non-null     object
 2               18 non-null     object
dtypes: float64(1), object(2)
memory usage: 698.0+ bytes
CPU times: user 439 ms, sys: 15.7 ms, total: 455 ms
Wall time: 454 ms


In [ ]:
%Checkpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high_small/checkpoints/post_cell_10_try_3.pickle